## Aula 4 - Arvores e RPN

Arvore binaria de busca, travessias, remocao e expressao RPN.


In [1]:
"""Exercicios da Aula 4 - arvores, buscas e RPN."""

from collections import deque


class Nodo:
    def __init__(self, valor):
        self.valor = valor
        self.esquerda = None
        self.direita = None


class BST:
    def __init__(self):
        self.raiz = None

    def inserir(self, valor):
        self.raiz = self._inserir(self.raiz, valor)

    def inserir_lista(self, valores):
        for valor in valores:
            self.inserir(valor)

    def _inserir(self, nodo, valor):
        if nodo is None:
            return Nodo(valor)

        if valor < nodo.valor:
            nodo.esquerda = self._inserir(nodo.esquerda, valor)
        elif valor > nodo.valor:
            nodo.direita = self._inserir(nodo.direita, valor)

        return nodo

    def em_ordem(self):
        valores = []
        self._em_ordem(self.raiz, valores)
        return valores

    def _em_ordem(self, nodo, valores):
        if nodo is None:
            return

        self._em_ordem(nodo.esquerda, valores)
        valores.append(nodo.valor)
        self._em_ordem(nodo.direita, valores)

    def pre_ordem(self):
        valores = []
        self._pre_ordem(self.raiz, valores)
        return valores

    def _pre_ordem(self, nodo, valores):
        if nodo is None:
            return

        valores.append(nodo.valor)
        self._pre_ordem(nodo.esquerda, valores)
        self._pre_ordem(nodo.direita, valores)

    def pos_ordem(self):
        valores = []
        self._pos_ordem(self.raiz, valores)
        return valores

    def _pos_ordem(self, nodo, valores):
        if nodo is None:
            return

        self._pos_ordem(nodo.esquerda, valores)
        self._pos_ordem(nodo.direita, valores)
        valores.append(nodo.valor)

    def bfs(self):
        if self.raiz is None:
            return []

        valores = []
        fila = deque([self.raiz])

        while fila:
            nodo = fila.popleft()
            valores.append(nodo.valor)

            if nodo.esquerda is not None:
                fila.append(nodo.esquerda)
            if nodo.direita is not None:
                fila.append(nodo.direita)

        return valores

    def remover(self, valor):
        self.raiz = self._remover(self.raiz, valor)

    def _remover(self, nodo, valor):
        if nodo is None:
            return None

        if valor < nodo.valor:
            nodo.esquerda = self._remover(nodo.esquerda, valor)
        elif valor > nodo.valor:
            nodo.direita = self._remover(nodo.direita, valor)
        else:
            if nodo.esquerda is None:
                return nodo.direita
            if nodo.direita is None:
                return nodo.esquerda

            sucessor = self._menor_nodo(nodo.direita)
            nodo.valor = sucessor.valor
            nodo.direita = self._remover(nodo.direita, sucessor.valor)

        return nodo

    def _menor_nodo(self, nodo):
        atual = nodo

        while atual.esquerda is not None:
            atual = atual.esquerda

        return atual


class ArvoreExpressaoRPN:
    operadores = {"+", "-", "*", "/"}

    @classmethod
    def construir(cls, expressao):
        pilha = []

        for token in expressao.split():
            if token in cls.operadores:
                direita = pilha.pop()
                esquerda = pilha.pop()
                nodo = Nodo(token)
                nodo.esquerda = esquerda
                nodo.direita = direita
                pilha.append(nodo)
            else:
                pilha.append(Nodo(float(token)))

        if len(pilha) != 1:
            raise ValueError("expressao RPN invalida")

        arvore = cls()
        arvore.raiz = pilha[0]
        return arvore

    def __init__(self):
        self.raiz = None

    def em_ordem(self):
        return self._em_ordem(self.raiz)

    def _em_ordem(self, nodo):
        if nodo is None:
            return ""

        if nodo.valor not in self.operadores:
            return str(nodo.valor)

        esquerda = self._em_ordem(nodo.esquerda)
        direita = self._em_ordem(nodo.direita)
        return f"({esquerda} {nodo.valor} {direita})"

    def avaliar(self):
        return self._avaliar(self.raiz)

    def _avaliar(self, nodo):
        if nodo.valor not in self.operadores:
            return nodo.valor

        esquerda = self._avaliar(nodo.esquerda)
        direita = self._avaliar(nodo.direita)

        if nodo.valor == "+":
            return esquerda + direita
        if nodo.valor == "-":
            return esquerda - direita
        if nodo.valor == "*":
            return esquerda * direita
        if nodo.valor == "/":
            return esquerda / direita

        raise ValueError("operador invalido")


def avaliar_rpn_com_pilha(expressao):
    pilha = []

    for token in expressao.split():
        if token in ArvoreExpressaoRPN.operadores:
            direita = pilha.pop()
            esquerda = pilha.pop()

            if token == "+":
                pilha.append(esquerda + direita)
            elif token == "-":
                pilha.append(esquerda - direita)
            elif token == "*":
                pilha.append(esquerda * direita)
            elif token == "/":
                pilha.append(esquerda / direita)
        else:
            pilha.append(float(token))

    if len(pilha) != 1:
        raise ValueError("expressao RPN invalida")

    return pilha[0]


## Exemplos de execucao


In [2]:
valores = [50, 30, 70, 20, 40, 60, 80]
arvore = BST()
arvore.inserir_lista(valores)

print("Em ordem:", arvore.em_ordem())
print("Pre ordem:", arvore.pre_ordem())
print("Pos ordem:", arvore.pos_ordem())
print("BFS:", arvore.bfs())

arvore.remover(80)
print("Depois de remover 80:", arvore.em_ordem())

expressao = "20 40 + 60 80 - *"
arvore_rpn = ArvoreExpressaoRPN.construir(expressao)
print("RPN:", expressao)
print("Infixa:", arvore_rpn.em_ordem())
print("Resultado pela arvore:", arvore_rpn.avaliar())
print("Resultado pela pilha:", avaliar_rpn_com_pilha(expressao))


Em ordem: [20, 30, 40, 50, 60, 70, 80]
Pre ordem: [50, 30, 20, 40, 70, 60, 80]
Pos ordem: [20, 40, 30, 60, 80, 70, 50]
BFS: [50, 30, 70, 20, 40, 60, 80]
Depois de remover 80: [20, 30, 40, 50, 60, 70]
RPN: 20 40 + 60 80 - *
Infixa: ((20.0 + 40.0) * (60.0 - 80.0))
Resultado pela arvore: -1200.0
Resultado pela pilha: -1200.0
